<a href="https://colab.research.google.com/github/rhans2001/anyname/blob/master/cifar10/cifar10_cnn.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
import torch.optim as optim
import matplotlib.pyplot as plt
import numpy as np

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

Device: cuda


In [5]:
# ResNet was trained on ImageNet which used these exact
# mean and std values — we must match them
train_transform = transforms.Compose([
    transforms.Resize(224),                  # ResNet expects 224x224
    transforms.RandomHorizontalFlip(),
    transforms.RandomCrop(224, padding=4),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],          # ImageNet mean per channel
        std= [0.229, 0.224, 0.225]           # ImageNet std per channel
    )
])

test_transform = transforms.Compose([
    transforms.Resize(224),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std= [0.229, 0.224, 0.225]
    )
])

train_data = torchvision.datasets.CIFAR10(root='data', train=True,  download=True, transform=train_transform)
test_data  = torchvision.datasets.CIFAR10(root='data', train=False, download=True, transform=test_transform)

train_loader = DataLoader(train_data, batch_size=32, shuffle=True,  num_workers=2)
test_loader  = DataLoader(test_data,  batch_size=32, shuffle=False, num_workers=2)

print(f"Training samples: {len(train_data)}")
print(f"Test samples:     {len(test_data)}")

100%|██████████| 170M/170M [00:13<00:00, 13.1MB/s]


Training samples: 50000
Test samples:     10000


In [6]:
import torchvision.models as models

# load ResNet18 with pretrained ImageNet weights
resnet = models.resnet18(weights='IMAGENET1K_V1')

# freeze all layers — don't touch the learned weights
for param in resnet.parameters():
    param.requires_grad = False

# replace the final layer — originally 1000 ImageNet classes
# with our own 10 class classifier
resnet.fc = nn.Linear(resnet.fc.in_features, 10)

# move to GPU
resnet = resnet.to(device)

# only train the final layer
optimizer = optim.Adam(resnet.fc.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss()

print(resnet.fc)
print(f"\nTotal parameters:     {sum(p.numel() for p in resnet.parameters()):,}")
print(f"Trainable parameters: {sum(p.numel() for p in resnet.parameters() if p.requires_grad):,}")

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 208MB/s]


Linear(in_features=512, out_features=10, bias=True)

Total parameters:     11,181,642
Trainable parameters: 5,130


In [7]:
num_epochs = 10
train_accs = []
test_accs  = []

for epoch in range(num_epochs):
    # training
    resnet.train()
    correct = 0
    total = 0
    running_loss = 0.0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = resnet(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    train_acc = 100 * correct / total
    train_accs.append(train_acc)

    # evaluation
    resnet.eval()
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = resnet(images)
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    test_acc = 100 * correct / total
    test_accs.append(test_acc)

    print(f"Epoch {epoch+1:2d}/{num_epochs} - Loss: {running_loss/len(train_loader):.4f} - Train: {train_acc:.2f}% - Test: {test_acc:.2f}%")

print("Training complete!")

Epoch  1/10 - Loss: 0.8016 - Train: 73.53% - Test: 78.93%
Epoch  2/10 - Loss: 0.6343 - Train: 78.00% - Test: 80.23%
Epoch  3/10 - Loss: 0.6080 - Train: 78.91% - Test: 79.56%
Epoch  4/10 - Loss: 0.6051 - Train: 79.16% - Test: 80.43%
Epoch  5/10 - Loss: 0.5929 - Train: 79.69% - Test: 80.06%
Epoch  6/10 - Loss: 0.5926 - Train: 79.47% - Test: 79.85%
Epoch  7/10 - Loss: 0.5888 - Train: 79.86% - Test: 80.36%
Epoch  8/10 - Loss: 0.5843 - Train: 79.76% - Test: 80.30%
Epoch  9/10 - Loss: 0.5845 - Train: 79.92% - Test: 80.04%
Epoch 10/10 - Loss: 0.5786 - Train: 80.17% - Test: 79.84%
Training complete!


In [8]:
# unfreeze layer4 — the deepest feature extraction block
for param in resnet.layer4.parameters():
    param.requires_grad = True

# check new trainable parameter count
print(f"Trainable parameters: {sum(p.numel() for p in resnet.parameters() if p.requires_grad):,}")

# use a lower learning rate — we're fine-tuning now, not learning from scratch
# too high a learning rate would destroy the carefully learned ImageNet weights
optimizer = optim.Adam([
    {'params': resnet.layer4.parameters(), 'lr': 0.0001},
    {'params': resnet.fc.parameters(),     'lr': 0.001}
], )

criterion = nn.CrossEntropyLoss()

Trainable parameters: 8,398,858


In [ ]:
num_epochs = 10
train_accs_ft = []
test_accs_ft  = []

for epoch in range(num_epochs):
    resnet.train()
    correct = 0
    total = 0
    running_loss = 0.0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = resnet(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    train_acc = 100 * correct / total
    train_accs_ft.append(train_acc)

    resnet.eval()
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = resnet(images)
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    test_acc = 100 * correct / total
    test_accs_ft.append(test_acc)

    print(f"Epoch {epoch+1:2d}/{num_epochs} - Loss: {running_loss/len(train_loader):.4f} - Train: {train_acc:.2f}% - Test: {test_acc:.2f}%")

print("Fine-tuning complete!")

Epoch  1/10 - Loss: 0.4364 - Train: 85.66% - Test: 89.65%
Epoch  2/10 - Loss: 0.2610 - Train: 91.25% - Test: 91.05%
Epoch  3/10 - Loss: 0.1991 - Train: 93.17% - Test: 91.13%
Epoch  4/10 - Loss: 0.1562 - Train: 94.59% - Test: 91.72%
Epoch  5/10 - Loss: 0.1252 - Train: 95.65% - Test: 91.53%
Epoch  6/10 - Loss: 0.1099 - Train: 96.20% - Test: 92.21%
Epoch  7/10 - Loss: 0.0929 - Train: 96.89% - Test: 93.04%
Epoch  8/10 - Loss: 0.0820 - Train: 97.18% - Test: 92.93%


In [2]:
epochs = list(range(1, 11))

v1_test    = [62.21, 70.31, 73.03, 73.88, 75.93, 76.43, 76.50, 76.59, 77.33, 77.65]
v2_test    = [64.98, 67.01, 72.89, 75.70, 76.82, 76.89, 76.95, 79.70, 79.15, 79.75]
resnet_test    = [77.66, 78.68, 80.48, 80.64, 79.98, 80.61, 80.09, 80.65, 80.26, 80.52]
resnet_ft_test = [89.10, 91.30, 90.24, 92.29, 92.10, 91.73, 92.28, 92.62, 92.33, 92.58]

plt.figure(figsize=(12, 6))
plt.plot(epochs, v1_test,       'r-o',  label='V1 — basic CNN')
plt.plot(epochs, v2_test,       'b-o',  label='V2 — augmented')
plt.plot(epochs, resnet_test,   'g-o',  label='ResNet frozen')
plt.plot(epochs, resnet_ft_test,'m-o',  label='ResNet finetuned')

plt.xlabel('Epoch')
plt.ylabel('Test Accuracy %')
plt.title('All Models — Test Accuracy Comparison')
plt.legend()
plt.grid(True)
plt.ylim(60, 100)
plt.show()

NameError: name 'plt' is not defined

In [3]:
from sklearn.metrics import confusion_matrix
import seaborn as sns

classes = ['airplane', 'automobile', 'bird', 'cat', 'deer',
           'dog', 'frog', 'horse', 'ship', 'truck']

# collect all predictions
all_predicted = []
all_actual = []

resnet.eval()
with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = resnet(images)
        _, predicted = torch.max(outputs, 1)
        all_predicted.extend(predicted.cpu().numpy())
        all_actual.extend(labels.cpu().numpy())

# confusion matrix
cm = confusion_matrix(all_actual, all_predicted)

# plot
plt.figure(figsize=(12, 10))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=classes,
            yticklabels=classes)
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('ResNet Finetuned — Confusion Matrix')
plt.tight_layout()
plt.show()

# mistakes per class
print(f"\nTotal mistakes: {sum(p != a for p, a in zip(all_predicted, all_actual))} out of 10,000")
print("\nMistakes per class:")
for i, cls in enumerate(classes):
    mistakes = sum(cm[i]) - cm[i][i]
    print(f"{cls:12s} → {mistakes} mistakes")

NameError: name 'resnet' is not defined

In [ ]:
# collect wrong predictions with images
wrong_images = []
wrong_predicted = []
wrong_actual = []

resnet.eval()
with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = resnet(images)
        _, predicted = torch.max(outputs, 1)

        wrong_mask = predicted != labels
        wrong_images.extend(images[wrong_mask].cpu().numpy())
        wrong_predicted.extend(predicted[wrong_mask].cpu().numpy())
        wrong_actual.extend(labels[wrong_mask].cpu().numpy())

# filter specifically dog mistakes
dog_idx = 5
dog_wrong_images    = [img for img, a in zip(wrong_images, wrong_actual) if a == dog_idx]
dog_wrong_predicted = [p   for p, a   in zip(wrong_predicted, wrong_actual) if a == dog_idx]
dog_wrong_actual    = [a   for a       in wrong_actual if a == dog_idx]

print(f"Total dog mistakes: {len(dog_wrong_images)}")
print("\nDogs confused as:")
for i in range(10):
    count = dog_wrong_predicted.count(i)
    if count > 0:
        print(f"  {classes[i]:12s} → {count} times")

# plot first 32 dog mistakes
fig, axes = plt.subplots(4, 8, figsize=(16, 8))
for i, ax in enumerate(axes.flat):
    if i < len(dog_wrong_images):
        img = np.transpose(dog_wrong_images[i], (1, 2, 0))
        # unnormalise from ImageNet values
        mean = np.array([0.485, 0.456, 0.406])
        std  = np.array([0.229, 0.224, 0.225])
        img  = std * img + mean
        img  = np.clip(img, 0, 1)
        ax.imshow(img)
        ax.set_title(f"P:{classes[dog_wrong_predicted[i]]}\nA:dog",
                     fontsize=7, color='red')
    ax.axis('off')

plt.suptitle("Dog mistakes — what did ResNet think they were?", y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# what is dog being confused WITH in ResNet?
dog_idx = 5
dog_row = cm[dog_idx]

print("When actual is DOG, model predicted:")
for i, count in enumerate(dog_row):
    if i != dog_idx:
        print(f"  {classes[i]:12s} → {count} times")
print(f"\nCorrect dog predictions: {dog_row[dog_idx]}")
print(f"Total dog mistakes:      {sum(dog_row) - dog_row[dog_idx]}")

In [ ]:
# what is cat being confused WITH in ResNet?
cat_idx = 3
cat_row = cm[cat_idx]

print("When actual is CAT, model predicted:")
for i, count in enumerate(cat_row):
    if i != cat_idx:
        print(f"  {classes[i]:12s} → {count} times")
print(f"\nCorrect cat predictions: {cat_row[cat_idx]}")
print(f"Total cat mistakes:      {sum(cat_row) - cat_row[cat_idx]}")

In [ ]:
import torch
from google.colab import drive
drive.mount('/content/drive')

# save finetuned ResNet
torch.save(resnet.state_dict(), '/content/drive/MyDrive/resnet_cifar10_finetuned.pth')
print("ResNet finetuned model saved!")

In [ ]:
print(type(resnet))